# Training on Colab's GPUs — *KON-Artist model*

In [ ]:
import os
import shutil

## Cloud connection

### Google Cloud Connection

In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


To copy a single file:
```Bash
!cp "/content/my_file.py" "/content/drive/MyDrive/Project_Folder/"
````

To copy an entire folder:
```Bash
!cp -r "/content/my_results_folder" "/content/drive/MyDrive/Project_Folder/"
```

### W&B API key

In [ ]:
%%writefile .env
WANDB_API_KEY=api/key/for/wandb

Writing .env


In [23]:
!pip install python-dotenv

In [ ]:
from dotenv import load_dotenv
import wandb

# Use override=True to make sure the new key from .env
# replaces the "wandb_v1_I" currently in memory.
load_dotenv(override=True)

# Verify length before logging in (should be 40)
key = os.getenv("WANDB_API_KEY")
print(f"Key loaded. Length: {len(key) if key else 0}")

if key and len(key) >= 40:
    wandb.login()
else:
    print("Error: Key is still too short. Check your .env file.")

Key loaded. Length: 86


## Cloning repositories

### Cloning own repository

In [25]:
!git clone -b colab https://github.com/MLsound/KON-Artist/

Cloning into 'KON-Artist'...
remote: Enumerating objects: 244, done.
remote: Counting objects: 100% (244/244), done.
remote: Compressing objects: 100% (175/175), done.
remote: Total 244 (delta 128), reused 170 (delta 60), pack-reused 0 (from 0)
Receiving objects: 100% (244/244), 1.18 MiB | 14.08 MiB/s, done.
Resolving deltas: 100% (128/128), done.


In [26]:
%cd /content/KON-Artist

[Errno 2] No such file or directory: '/content/KON-Artist'
/content/KON-Artist


In [ ]:
ROOT_PATH = "/content/KON-Artist"

In [27]:
!ls

01_AASIST_basic.ipynb	    condacolab_install.log  pytest.ini	tests
02_AASIST_evaluation.ipynb  environment.yml	    README.md	third_party
03_DSP_pipeline.ipynb	    KON-Artist		    root.py	wandb
04_RL_cycle.ipynb	    outputs		    scripts	weights
05_Colab_training.ipynb     pyproject.toml	    src


### Cloning AASIST3 model repo

In [28]:
!git clone https://github.com/mtuciru/AASIST3.git third_party/AASIST3

fatal: destination path 'third_party/AASIST3' already exists and is not an empty directory.


## Creating environment

In [29]:
!pip install -q condacolab
import condacolab
condacolab.install()

✨🍰✨ Everything looks OK!


In [30]:
!conda init

no change     /usr/local/condabin/conda
no change     /usr/local/bin/conda
no change     /usr/local/bin/conda-env
no change     /usr/local/bin/activate
no change     /usr/local/bin/deactivate
no change     /usr/local/etc/profile.d/conda.sh
no change     /usr/local/etc/fish/conf.d/conda.fish
no change     /usr/local/shell/condabin/Conda.psm1
no change     /usr/local/shell/condabin/conda-hook.ps1
no change     /usr/local/lib/python3.11/site-packages/xontrib/conda.xsh
no change     /usr/local/etc/profile.d/conda.csh
no change     /root/.bashrc
No action taken.


In [31]:
# Run your training or testing script
!source /usr/local/bin/activate

In [32]:
!pip install --quiet \
    numpy==1.26.4 \
    scipy==1.13.1 \
    pandas==2.3.3 \
    torch==2.2.2 torchaudio==2.2.2 torchvision==0.17.2 \
    stable-baselines3==2.4.1 gymnasium==1.0.0 \
    librosa==0.11.0 soundfile==0.13.1 \
    wandb==0.25.1 matplotlib==3.10.8 \
    transformers==4.40.0 datasets==2.19.1 huggingface-hub==0.36.2

In [33]:
import os
os.environ['MPLBACKEND'] = 'Agg'

In [34]:
!mkdir outputs/
!mkdir outputs/history/

mkdir: cannot create directory ‘outputs/’: File exists
mkdir: cannot create directory ‘outputs/history/’: File exists


## Training cycle

In [35]:
from src.utils.misc import create_timestamp
timestamp = create_timestamp()

In [36]:
EXPORT_FOLDER = f"KONArtist_{timestamp}"

In [37]:
# 2. Run with PYTHONPATH set to the current directory (.)
!PYTHONPATH=. python -m scripts.train

Streaming output truncated to the last 5000 lines.
[2026-04-22 17:10:22] INFO [audio_attack:115] Step 6 | DSP Params: {'jitter': 1.0, 'shimmer': -0.5909477, 'tilt': 0.3980918, 'harmonics': 1.0, 'threshold': -32.425382137298584, 'ratio': 8.332041263580322, 'bitrate': 32000} | Reward: 7.32000984271508e-08
[2026-04-22 17:10:22] INFO [audio_attack:115] Step 7 | DSP Params: {'jitter': 1.0, 'shimmer': -1.0, 'tilt': 1.0, 'harmonics': -0.9217758, 'threshold': -60.0, 'ratio': 11.0, 'bitrate': 119898} | Reward: 3.8215952372411266e-05
[2026-04-22 17:10:22] INFO [audio_attack:115] Step 8 | DSP Params: {'jitter': 1.0, 'shimmer': 1.0, 'tilt': 1.0, 'harmonics': 1.0, 'threshold': -60.0, 'ratio': 1.0, 'bitrate': 160000} | Reward: 5.289993509904889e-07
[2026-04-22 17:10:22] INFO [audio_attack:115] Step 9 | DSP Params: {'jitter': 1.0, 'shimmer': -0.27738613, 'tilt': 1.0, 'harmonics': 1.0, 'threshold': -27.821983098983765, 'ratio': 11.0, 'bitrate': 75575} | Reward: 2.9814398658345453e-05
[2026-04-22 17:10

## Saving results into cloud

In [ ]:
print("Saving into cloud folder:", EXPORT_FOLDER)

KONArtist_20260422_131706


In [ ]:
# Set the source and destination paths
# Origin in local filesystem
source_path = f"{ROOT_PATH}/outputs"
# Destination in Google Drive
destination_path = f"/content/drive/MyDrive/{EXPORT_FOLDER}"

# Move the folder from local to Drive
try:
    if not os.path.exists(destination_path):
        os.makedirs(destination_path)
        print(f"Carpeta creada: {destination_path}")
    # shutil.move manages the copying and deletion of the original
    # even between different filesystems (from local to Drive)
    shutil.move(source_path, destination_path)
    print(f"Se ha movido correctamente a: {destination_path}")
except FileNotFoundError:
    print(f"Error: No se encontró la carpeta de origen en {source_path}")
except Exception as e:
    print(f"Ocurrió un error inesperado: {e}")

Se ha movido correctamente a: /content/drive/MyDrive/KONArtist_20260422_131706


In [40]:
# Example: Creating a folder for your project
#%mkdir -p "/content/drive/MyDrive/"
#%cp -r "/content/KON-Artist/outputs" "/content/drive/MyDrive/"